# 4-Stroke Engine Fins में Heat Convection — Fluid Dynamics Notebook
**Languages:** Python, C, C++, Rust, Go, Bash, HTML  
**Topics:** Fin analysis, Data Science, Cleaning, Visualization, Machine Learning, Deep Learning, Computer Vision

Physics: Fin equation `θ'' − m²θ = 0`, `m = √(hP/kA)`; h निकालने के लिए flat-plate forced convection (Nu–Re–Pr correlations)।

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, subprocess, shutil, os
from IPython.display import HTML, display
np.random.seed(42)
# Engine fin parameters (aluminium, rectangular fin)
k=180.0; Tb=200.0; Tinf=40.0          # W/mK, °C, °C
L=0.05; w=0.10; t=0.003               # m: height, width, thickness
P=2*(w+t); Ac=w*t
def air_props(T):                      # approx. air properties at film temp (°C)
    Tk=T+273.15
    nu=1.516e-5*(Tk/293.15)**1.75*(1.0)  # m2/s (approx)
    kf=0.0263*(Tk/293.15)**0.8
    return nu,kf,0.707
def h_forced(V,Lc=0.10):
    nu,kf,Pr=air_props((Tb+Tinf)/2)
    Re=V*Lc/nu
    Nu=0.664*Re**0.5*Pr**(1/3) if Re<5e5 else (0.037*Re**0.8-871)*Pr**(1/3)
    return Nu*kf/Lc,Re
def fin(h,k=k,L=L,t=t):
    P=2*(w+t);Ac=w*t
    m=np.sqrt(h*P/(k*Ac)); Lc=L+t/2
    Q=np.sqrt(h*P*k*Ac)*(Tb-Tinf)*np.tanh(m*Lc)
    eta=np.tanh(m*Lc)/(m*Lc); return Q,eta,m
for V in [1,5,10,20,30]:
    h,Re=h_forced(V); Q,e,m=fin(h); print(f"V={V:3d} m/s  Re={Re:9.0f}  h={h:6.1f}  Q={Q:6.1f} W  eta={e:.3f}")

## 1) Data Science: Dataset बनाना (noise, missing values, outliers के साथ)

In [ ]:
N=1500
V=np.random.uniform(0.5,35,N); Lf=np.random.uniform(0.02,0.08,N)
tf=np.random.uniform(0.002,0.006,N); kf_=np.random.choice([120,180,230],N)
rows=[]
for v,l,tt,kk in zip(V,Lf,tf,kf_):
    h,Re=h_forced(v); Q,e,m=fin(h,kk,l,tt); rows.append((v,l,tt,kk,Re,h,Q*(1+np.random.normal(0,0.03)),e))
df=pd.DataFrame(rows,columns=["V","L","t","k","Re","h","Q","eta"])
df.loc[np.random.choice(N,60,replace=False),"V"]=np.nan
df.loc[np.random.choice(N,40,replace=False),"Q"]*=5      # outliers
df.to_csv("fin_data_raw.csv",index=False); df.describe()

## 2) Data Cleaning (pandas)

In [ ]:
d=pd.read_csv("fin_data_raw.csv")
print("Missing:\n",d.isna().sum()[d.isna().sum()>0])
d["V"]=d["V"].fillna(d["V"].median())
# IQR outlier removal on Q relative to model
d["ratio"]=d["Q"]/ (d["Q"].rolling(25,center=True,min_periods=1).median())
q1,q3=d["Q"].quantile([.25,.75]); iqr=q3-q1
clean=d[(d["Q"]>q1-3*iqr)&(d["Q"]<q3+3*iqr)].drop(columns="ratio").reset_index(drop=True)
print(len(d),"->",len(clean)); clean.to_csv("fin_data_clean.csv",index=False)

## 3) Data Visualization

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
Vs=np.linspace(0.5,35,60); hs=[h_forced(v)[0] for v in Vs]
ax[0].plot(Vs,hs); ax[0].set(title="h vs Air Velocity",xlabel="V (m/s)",ylabel="h (W/m²K)")
for kk in [120,180,230]:
    ax[1].plot(Vs,[fin(h_forced(v)[0],kk)[0] for v in Vs],label=f"k={kk}")
ax[1].set(title="Fin Heat Rate",xlabel="V (m/s)",ylabel="Q (W)"); ax[1].legend()
sc=ax[2].scatter(clean.V,clean.eta,c=clean.L,s=6,cmap="inferno"); plt.colorbar(sc,label="L (m)")
ax[2].set(title="Fin Efficiency",xlabel="V",ylabel="eta")
plt.tight_layout(); plt.savefig("fin_plots.png",dpi=120); plt.show()
print(clean[["V","L","t","k","h","Q","eta"]].corr().round(2))

## 4) Machine Learning — Q predict करना

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
X=clean[["V","L","t","k"]]; y=clean["Q"]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=0)
for name,mdl in [("RandomForest",RandomForestRegressor(200,random_state=0)),("GradBoost",GradientBoostingRegressor(random_state=0))]:
    mdl.fit(Xtr,ytr); p=mdl.predict(Xte); print(f"{name}: R2={r2_score(yte,p):.4f}  MAE={mean_absolute_error(yte,p):.2f} W")
print(dict(zip(X.columns,RandomForestRegressor(200,random_state=0).fit(Xtr,ytr).feature_importances_.round(3))))

## 5) Deep Learning — Neural Network (PyTorch हो तो वो, नहीं तो sklearn MLP)

In [ ]:
from sklearn.preprocessing import StandardScaler
sx,sy=StandardScaler().fit(Xtr),StandardScaler().fit(ytr.values.reshape(-1,1))
A,B=sx.transform(Xtr),sy.transform(ytr.values.reshape(-1,1)).ravel()
try:
    import torch,torch.nn as nn
    net=nn.Sequential(nn.Linear(4,64),nn.ReLU(),nn.Linear(64,64),nn.ReLU(),nn.Linear(64,1))
    opt=torch.optim.Adam(net.parameters(),1e-2); At=torch.tensor(A,dtype=torch.float32); Bt=torch.tensor(B,dtype=torch.float32).view(-1,1)
    for ep in range(300):
        opt.zero_grad(); loss=nn.functional.mse_loss(net(At),Bt); loss.backward(); opt.step()
    pr=net(torch.tensor(sx.transform(Xte),dtype=torch.float32)).detach().numpy()
    print("PyTorch NN")
except ImportError:
    from sklearn.neural_network import MLPRegressor
    net=MLPRegressor(hidden_layer_sizes=(64,64),max_iter=2000,random_state=0).fit(A,B); pr=net.predict(sx.transform(Xte)).reshape(-1,1); print("sklearn MLP")
pr=sy.inverse_transform(pr.reshape(-1,1)).ravel(); print("R2:",round(r2_score(yte,pr),4))

## 6) Multi-language: 1D Fin Finite-Difference Solver (C, C++, Rust, Go, Python)
सब same algorithm (Thomas tridiagonal) चलाते हैं; Q compare होगा।

In [ ]:
%%writefile fin.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
int main(int c,char**v){double h=atof(v[1]),k=180,L=0.05,w=0.1,t=0.003,Tb=200,Ti=40;
int N=200;double P=2*(w+t),A=w*t,m2=h*P/(k*A),dx=L/N,th0=Tb-Ti;
double a[201],b[201],cc[201],d[201],x[201];
b[0]=1;cc[0]=0;d[0]=th0;a[0]=0;
for(int i=1;i<N;i++){a[i]=1;b[i]=-(2+m2*dx*dx);cc[i]=1;d[i]=0;}
a[N]=2;b[N]=-(2+m2*dx*dx+2*dx*h/k);cc[N]=0;d[N]=0;
for(int i=1;i<=N;i++){double f=a[i]/b[i-1];b[i]-=f*cc[i-1];d[i]-=f*d[i-1];}
x[N]=d[N]/b[N];for(int i=N-1;i>=0;i--)x[i]=(d[i]-cc[i]*x[i+1])/b[i];
printf("%.4f\n",-k*A*(-3*x[0]+4*x[1]-x[2])/(2*dx));return 0;}

In [ ]:
%%writefile fin.cpp
#include <iostream>
#include <vector>
#include <cmath>
#include <cstdlib>
using namespace std;
int main(int c,char**v){double h=atof(v[1]),k=180,L=0.05,w=0.1,t=0.003,Tb=200,Ti=40;
int N=200;double P=2*(w+t),A=w*t,m2=h*P/(k*A),dx=L/N;
vector<double> a(N+1,1),b(N+1,-(2+m2*dx*dx)),cc(N+1,1),d(N+1,0),x(N+1);
b[0]=1;cc[0]=0;d[0]=Tb-Ti;a[0]=0;a[N]=2;b[N]=-(2+m2*dx*dx+2*dx*h/k);cc[N]=0;
for(int i=1;i<=N;i++){double f=a[i]/b[i-1];b[i]-=f*cc[i-1];d[i]-=f*d[i-1];}
x[N]=d[N]/b[N];for(int i=N-1;i>=0;i--)x[i]=(d[i]-cc[i]*x[i+1])/b[i];
cout.precision(4);cout<<fixed<<-k*A*(-3*x[0]+4*x[1]-x[2])/(2*dx)<<endl;}

In [ ]:
%%writefile fin.rs
use std::env;
fn main(){let h:f64=env::args().nth(1).unwrap().parse().unwrap();
let (k,l,w,t,tb,ti)=(180.0f64,0.05f64,0.1f64,0.003f64,200.0f64,40.0f64);
let n=200usize;let p=2.0*(w+t);let a_c=w*t;let m2=h*p/(k*a_c);let dx=l/n as f64;
let mut a=vec![1.0;n+1];let mut b=vec![-(2.0+m2*dx*dx);n+1];let mut cc=vec![1.0;n+1];let mut d=vec![0.0;n+1];let mut x=vec![0.0;n+1];
b[0]=1.0;cc[0]=0.0;d[0]=tb-ti;a[0]=0.0;a[n]=2.0;b[n]=-(2.0+m2*dx*dx+2.0*dx*h/k);cc[n]=0.0;
for i in 1..=n{let f=a[i]/b[i-1];b[i]-=f*cc[i-1];d[i]-=f*d[i-1];}
x[n]=d[n]/b[n];for i in (0..n).rev(){x[i]=(d[i]-cc[i]*x[i+1])/b[i];}
println!("{:.4}",-k*a_c*(-3.0*x[0]+4.0*x[1]-x[2])/(2.0*dx));}

In [ ]:
%%writefile fin.go
package main
import ("fmt";"os";"strconv")
func main(){h,_:=strconv.ParseFloat(os.Args[1],64)
k,L,w,t,Tb,Ti:=180.0,0.05,0.1,0.003,200.0,40.0
N:=200;P:=2*(w+t);A:=w*t;m2:=h*P/(k*A);dx:=L/float64(N)
a:=make([]float64,N+1);b:=make([]float64,N+1);cc:=make([]float64,N+1);d:=make([]float64,N+1);x:=make([]float64,N+1)
for i:=range a{a[i]=1;b[i]=-(2+m2*dx*dx);cc[i]=1}
b[0]=1;cc[0]=0;d[0]=Tb-Ti;a[0]=0;a[N]=2;b[N]=-(2+m2*dx*dx+2*dx*h/k);cc[N]=0
for i:=1;i<=N;i++{f:=a[i]/b[i-1];b[i]-=f*cc[i-1];d[i]-=f*d[i-1]}
x[N]=d[N]/b[N];for i:=N-1;i>=0;i--{x[i]=(d[i]-cc[i]*x[i+1])/b[i]}
fmt.Printf("%.4f\n",-k*A*(-3*x[0]+4*x[1]-x[2])/(2*dx))}

In [ ]:
%%bash
H=60
echo "== Bash: tools check =="; for t in gcc g++ rustc go; do command -v $t >/dev/null && echo "$t OK" || echo "$t NOT FOUND (skip)"; done
command -v gcc  >/dev/null && gcc -O2 fin.c -o fin_c -lm && echo "C   : $(./fin_c $H) W"
command -v g++  >/dev/null && g++ -O2 fin.cpp -o fin_cpp && echo "C++ : $(./fin_cpp $H) W"
command -v rustc>/dev/null && rustc -O fin.rs -o fin_rs 2>/dev/null && echo "Rust: $(./fin_rs $H) W"
command -v go   >/dev/null && go run fin.go $H | sed 's/^/Go  : /;s/$/ W/'

In [ ]:
# Python version + analytical check
def fd_python(h,N=200):
    m2=h*P/(k*Ac); dx=L/N; M=np.zeros((N+1,N+1)); r=np.zeros(N+1)
    M[0,0]=1; r[0]=Tb-Tinf
    for i in range(1,N): M[i,i-1:i+2]=[1,-(2+m2*dx*dx),1]
    M[N,N-1]=2; M[N,N]=-(2+m2*dx*dx+2*dx*h/k)
    x=np.linalg.solve(M,r); return -k*Ac*(-3*x[0]+4*x[1]-x[2])/(2*dx),x
Qfd,theta=fd_python(60); print("Python FD:",round(Qfd,4),"W | Analytical (adiabatic tip, corrected L):",round(fin(60)[0],4),"W")
plt.plot(np.linspace(0,L,201)*1000,theta+Tinf); plt.xlabel("Fin length (mm)"); plt.ylabel("T (°C)"); plt.title("Fin Temperature Profile"); plt.grid(); plt.show()

## 7) 2D Temperature Field + Computer Vision (hot-spot detection, edges)

In [ ]:
from scipy.ndimage import sobel, label
nx,ny=120,60; T=np.full((ny,nx),Tinf); h=60; dx=L/nx*2
T[:,0]=Tb
for it in range(6000):                     # Jacobi iteration, convective loss term
    Tn=T.copy(); Bi=h*dx/k
    Tn[1:-1,1:-1]=(T[2:,1:-1]+T[:-2,1:-1]+T[1:-1,2:]+T[1:-1,:-2]+Bi*Tinf*0.3)/(4+Bi*0.3)
    Tn[:,0]=Tb; Tn[0,:]=Tn[1,:]; Tn[-1,:]=Tn[-2,:]; Tn[:,-1]=Tn[:,-2]; T=Tn
img=((T-T.min())/(T.max()-T.min())*255).astype(np.uint8)
gx,gy=sobel(img.astype(float),1),sobel(img.astype(float),0); edges=np.hypot(gx,gy)
hot=img>170; lab,nlab=label(hot)
try:
    import cv2; col=cv2.applyColorMap(img,cv2.COLORMAP_JET)[:,:,::-1]; print("OpenCV used")
except ImportError: col=plt.cm.jet(img)[:,:,:3]
fig,ax=plt.subplots(1,3,figsize=(15,4)); ax[0].imshow(col); ax[0].set_title("Temperature (JET)")
ax[1].imshow(edges,cmap="gray"); ax[1].set_title("Gradient / Edges"); ax[2].imshow(hot,cmap="hot"); ax[2].set_title(f"Hot region (>170 gray), blobs={nlab}")
plt.show()

## 8) HTML Report

In [ ]:
rows="".join(f"<tr><td>{v}</td><td>{h_forced(v)[0]:.1f}</td><td>{fin(h_forced(v)[0])[0]:.1f}</td><td>{fin(h_forced(v)[0])[1]:.3f}</td></tr>" for v in [2,5,10,20,30])
display(HTML(f'''<div style="font-family:sans-serif"><h3 style="color:#c0392b">Engine Fin Convection Summary</h3>
<table border=1 cellpadding=6 style="border-collapse:collapse"><tr style="background:#eee"><th>V (m/s)</th><th>h (W/m²K)</th><th>Q (W)</th><th>η</th></tr>{rows}</table></div>'''))

## 9) Upgrade: 3D diagrams, comparison tables, advanced analyses
इस हिस्से के लिए repo की `.py` फाइलें (flat folder में) चाहिए: `fin_core.py`, `engine_fin.py`, `engine_fin_advanced.py`।

In [ ]:
%run engine_fin.py
%run engine_fin_advanced.py

In [ ]:
from IPython.display import Image, display, Markdown
import glob
for f in sorted(glob.glob("fig*.png")): display(Markdown(f"**{f}**")); display(Image(f, width=700))

In [ ]:
import pandas as pd
for f in sorted(glob.glob("table_*.csv")): display(Markdown(f"### {f}")); display(pd.read_csv(f))

In [ ]:
# Live 3D surface inside the notebook: annular fin efficiency vs h and height
from fin_core import annular_eta
Hg,Lg=np.meshgrid(np.linspace(10,150,40),np.linspace(.01,.06,40))
Eg=np.array([[annular_eta(h,180,.003,.04,l) for h,l in zip(a,b)] for a,b in zip(Hg,Lg)])
fig=plt.figure(figsize=(7,5)); ax=fig.add_subplot(111,projection="3d"); ax.plot_surface(Hg,Lg*1000,Eg,cmap="viridis")
ax.set(xlabel="h",ylabel="height (mm)",zlabel="eta"); plt.show()

## 10) Upgrade 2: fin shapes, pressure drop, optimisation, animation
`engine_fin_optimization.py` चलाने पर नई figures, tables और GIF बनते हैं।

In [ ]:
%run engine_fin_optimization.py

In [ ]:
from IPython.display import Image, display
display(Image("anim_fin_transient.gif"))
for f in ["fig23_fin_shapes_comparison.png","fig24_thermal_hydraulic.png","fig25_pareto_optimisation.png","fig26_3d_pareto.png","fig27_3d_fin_shapes.png"]: display(Image(f,width=750))
import pandas as pd
for f in ["table_fin_shapes.csv","table_thermal_hydraulic.csv","table_pareto.csv"]: display(pd.read_csv(f))